# 04. Structured Output with LLMs

In this notebook, we will learn:
1. How to define custom data schemas using Pydantic `BaseModel`.
2. How to extract structured outputs from LLMs using `.with_structured_output()`.
3. How to use `JsonOutputParser` for JSON schema responses.

In [ ]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Load environment variables
load_dotenv()

### 1. Defining a Pydantic Schema

In [ ]:
class MovieReview(BaseModel):
    title: str = Field(description="The title of the movie")
    genre: List[str] = Field(description="List of genres for the movie")
    rating: float = Field(description="Rating out of 10")
    summary: str = Field(description="A 2-sentence summary of the movie plot")
    sentiment: str = Field(description="Overall sentiment: Positive, Negative, or Neutral")

# Instantiate LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# Bind structured output schema to LLM
structured_llm = llm.with_structured_output(MovieReview)

### 2. Extracting Structured Data from Unstructured Text

In [ ]:
review_text = """
Christopher Nolan's 'Inception' is a mind-bending sci-fi action thriller released in 2010. 
Dom Cobb is a skilled thief who steals corporate secrets using dream-sharing technology. 
He is given a chance to have his criminal history erased as payment for the implantation 
of another person's idea into the target's subconscious. The visual effects are breathtaking 
and Hans Zimmer's score is legendary. Easily 9.5 out of 10 stars!
"""

result: MovieReview = structured_llm.invoke(review_text)
print(f"Title: {result.title}")
print(f"Genres: {result.genre}")
print(f"Rating: {result.rating}")
print(f"Sentiment: {result.sentiment}")
print(f"Summary: {result.summary}")

### 3. Alternative: Using `JsonOutputParser`

In [ ]:
class PersonInfo(BaseModel):
    name: str = Field(description="Person's full name")
    skills: List[str] = Field(description="List of technical skills")
    experience_years: int = Field(description="Years of experience")

parser = JsonOutputParser(pydantic_object=PersonInfo)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract person information into JSON matching the instructions.\n{format_instructions}"),
    ("user", "{input}")
])

chain = prompt | llm | parser

person_data = chain.invoke({
    "input": "Debasish is a Software Engineer with 3 years of experience in Python, LangChain, and Streamlit.",
    "format_instructions": parser.get_format_instructions()
})

print(person_data)